# StockLLM V2：本地事件抽取與人工驗證
依序執行下列儲存格。使用 CPU，不需要 API key。
4 筆示例是合成英文新聞；這不是完整 E0–E4 回測。
免費 Colab 有資源限制，結束前務必下載結果。
完整欄位說明請看 [中文指南](https://github.com/yoyocar2333/stockllm-pit-news/blob/research/event-extraction-e0-e4/docs/EVENT_EXTRACTION_GUIDE_ZH.md)。


## 1. 取得研究分支
新 runtime 執行一次；重啟後只需要重新切換目錄。

In [ ]:
!git clone --branch research/event-extraction-e0-e4 --single-branch https://github.com/yoyocar2333/stockllm-pit-news.git /content/stockllm-pit-news
%cd /content/stockllm-pit-news


## 2. 安裝依賴與模型
若提示重新啟動工作階段，重啟後執行 `%cd /content/stockllm-pit-news` 再繼續。

In [ ]:
%pip install -r requirements-event.txt
!python -m spacy download en_core_web_sm


## 3. 執行示例與測試
每個輸出應有 4 列新聞，另附版本紀錄。

In [ ]:
!python -m event_extraction.run --input event_extraction/sample_news.csv --backend spacy --source title --output outputs/events_title.csv
!python -m event_extraction.run --input event_extraction/sample_news.csv --backend spacy --source full --output outputs/events_full.csv
!python -m pytest tests/test_event_extractors.py tests/test_event_annotation.py tests/test_event_run.py -q


In [ ]:
import json
import pandas as pd
events = pd.read_csv('outputs/events_title.csv', dtype=str, keep_default_na=False)
display(events[['article_id', 'title', 'event_count']])
display(pd.json_normalize(json.loads(events.iloc[0]['events_json'])))


## 4. 匯出人工檢查表
除錯用 assisted 表會顯示模型答案。下載後在 Excel / Google Sheets 填 gold 欄位、decision、evidence、review_status。
正式獨立驗證改用 `--blind`；盲標不顯示模型答案。請先閱讀指南第 5 節。
同名表已存在時程式會拒絕覆蓋，避免丟失人工標註。


In [ ]:
!python -m event_extraction.annotate prepare --input outputs/events_title.csv --output outputs/annotation_title.csv
!python -m event_extraction.annotate prepare --input outputs/events_full.csv --output outputs/annotation_full.csv
from google.colab import files
files.download('outputs/annotation_title.csv')
files.download('outputs/annotation_full.csv')


## 5. 人工填完後上傳並驗證
這一格請等人工完成後再執行，不要「全部執行」跳過標註。
只上傳一個已完成的標題標註 CSV；所有 pending 必須改成 reviewed 或 uncertain。
全文標註請另外改成對應的 predictions、annotations 與 output 路徑。


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
assert len(uploaded) == 1, '請只選一個已完成的標題標註 CSV'
Path('outputs/annotation_title_reviewed.csv').write_bytes(next(iter(uploaded.values())))
!python -m event_extraction.annotate validate --predictions outputs/events_title.csv --annotations outputs/annotation_title_reviewed.csv --output outputs/extraction_metrics_title.json


## 6. 下載結果
precision/recall/F1 是抽取一致性，不是股價預測績效。
檢查排除的 uncertain 篇數，再保存結果。


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/event_extraction_outputs', 'zip', 'outputs')
files.download('/content/event_extraction_outputs.zip')


## 選用：換成真實新聞
將五個必需欄位 article_id,ticker,target_name,title,content 準備成 UTF-8 CSV。
先用固定的 100–200 篇，同一批新聞同時有標題／內文，再比較兩種來源。


In [ ]:
from google.colab import files
from pathlib import Path
uploaded_news = files.upload()
assert len(uploaded_news) == 1, '請只選一個新聞 CSV'
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/raw/my_news.csv').write_bytes(next(iter(uploaded_news.values())))
!python -m event_extraction.run --input data/raw/my_news.csv --backend spacy --source title --output outputs/my_events_title.csv
!python -m event_extraction.run --input data/raw/my_news.csv --backend spacy --source full --output outputs/my_events_full.csv
!python -m event_extraction.annotate prepare --input outputs/my_events_title.csv --output outputs/gold_title_blind.csv --blind
files.download('outputs/gold_title_blind.csv')


## 選用：GLiNER2 本地比較
需先安裝上方 spaCy 依賴及英文模型。首次下載權重可能較久；不需要 API key。
GLiNER2 的 action 使用 spaCy 統一動詞原形。品質仍需同一份獨立人工 gold 驗證。


In [ ]:
%pip install "gliner2[local]==2.1.0"
!python -m event_extraction.run --backend gliner2 --source title --output outputs/gliner2_title.csv
